# Module 18 — Writing Professional Python

**Part IV: Advanced** · Level: Advanced · Prerequisites: Modules 1–17

## What you'll learn

- PEP 8 style and naming
- Docstrings that tools and teammates can use
- Pythonic idioms (and when *not* to use them)
- Refactoring toward small, reusable functions
- Logging instead of `print` for diagnostics
- Tooling that keeps code healthy

---

## 18.1 PEP 8: style that everyone shares

**PEP 8** is Python's style guide. Consistent style makes code easier to read, review, and maintain. The essentials:

- 4 spaces per indentation level; never mix tabs and spaces
- `snake_case` for functions and variables, `PascalCase` for classes, `UPPER_CASE` for constants
- Lines up to about 79–99 characters
- Two blank lines between top-level definitions, one between methods
- Spaces around operators (`a = b + c`), none inside brackets (`f(x)`, `[1, 2]`)
- Imports at the top, grouped: standard library, third party, local

In [ ]:
# Hard to read
def f(a,b):return a*b+1
X=f( 2,3 )

# Clear
def scale_and_shift(value, factor):
    return value * factor + 1

RESULT = scale_and_shift(2, 3)
print(RESULT)

## 18.2 Meaningful names

Names are the most important documentation you write.

In [ ]:
# Cryptic
def p(l):
    t = 0
    for i in l:
        t += i[1] * i[2]
    return t

# Descriptive
def order_total(line_items):
    """Return the total cost of (name, quantity, unit_price) line items."""
    total = 0.0
    for _name, quantity, unit_price in line_items:
        total += quantity * unit_price
    return total

print(order_total([("pen", 3, 1.5), ("ink", 1, 7.25)]))

## 18.3 Docstrings and comments

- A **docstring** says *what* a function, class, or module does, its inputs, outputs, and errors.
- A **comment** explains *why* something non-obvious is done. Do not restate the code.

In [ ]:
def retry_delay(attempt: int, base: float = 0.5, cap: float = 30.0) -> float:
    """Return how long to wait before retry number ``attempt``.

    Uses exponential backoff so a struggling service is not hammered.

    Args:
        attempt: The retry number, starting at 1.
        base: Delay for the first retry, in seconds.
        cap: Maximum delay, in seconds.

    Returns:
        The delay in seconds.

    Raises:
        ValueError: If ``attempt`` is less than 1.
    """
    if attempt < 1:
        raise ValueError("attempt must be >= 1")
    # Doubling each time; the cap stops delays from growing without limit.
    return min(cap, base * 2 ** (attempt - 1))

print([retry_delay(n) for n in range(1, 9)])

## 18.4 Pythonic idioms

In [ ]:
# Iterate directly, not by index
names = ["ada", "grace"]
for name in names:
    print(name)
for i, name in enumerate(names, start=1):
    print(i, name)

# Test emptiness by truthiness
if not names:
    print("nothing")

# Compare to None with `is`
value = None
if value is None:
    print("unset")

# Swap and multiple assignment
a, b = 1, 2
a, b = b, a

# Dictionary with default instead of if/else
config = {"retries": 3}
timeout = config.get("timeout", 30)

# join instead of += in loops
print(", ".join(str(n) for n in range(5)))

# Chained comparison
x = 5
print(0 < x < 10)

# Underscore for unused values
for _ in range(2):
    print("tick")

### EAFP vs LBYL

- **LBYL** — "Look before you leap": check, then act.
- **EAFP** — "Easier to ask forgiveness than permission": try, then handle failure. Often more Pythonic, and free of race conditions.

In [ ]:
data = {"a": 1}

# LBYL
if "b" in data:
    result = data["b"]
else:
    result = 0

# EAFP
try:
    result = data["b"]
except KeyError:
    result = 0

# Even better here:
result = data.get("b", 0)
print(result)

## 18.5 Refactoring toward reusable code

Good functions are **small, focused, and free of hidden state**. Compare:

In [ ]:
# Before: one function does parsing, validation, computation, and printing
def process(text):
    nums = []
    for part in text.split(","):
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    total = 0
    for n in nums:
        total += n
    print("Total:", total, "Average:", total / len(nums))

In [ ]:
# After: each function has one job and can be tested alone
def parse_numbers(text: str) -> list[int]:
    """Extract the integer fields from a comma-separated string."""
    return [int(part) for part in (p.strip() for p in text.split(",")) if part.isdigit()]

def summarize(numbers: list[int]) -> dict[str, float]:
    """Return the total and average of ``numbers``."""
    if not numbers:
        raise ValueError("no numbers to summarize")
    total = sum(numbers)
    return {"total": total, "average": total / len(numbers)}

def format_summary(summary: dict[str, float]) -> str:
    return f"Total: {summary['total']}  Average: {summary['average']:.2f}"

print(format_summary(summarize(parse_numbers("4, 8, x, 15, 16"))))

Rules of thumb:

1. If a function name needs the word "and", it probably does two things.
2. Separate **calculating** from **printing** and from **reading input**.
3. Prefer returning values over modifying globals.
4. Duplicate a block once? Fine. Three times? Extract a function.

## 18.6 Logging instead of `print`

`print` is for program output. `logging` is for diagnostics: it has levels, timestamps, destinations, and can be turned up or down without editing code.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s: %(message)s",
    force=True,
)
log = logging.getLogger("inventory")

def reserve(stock: int, wanted: int) -> int:
    log.debug("reserve called with stock=%s wanted=%s", stock, wanted)   # hidden at INFO
    if wanted > stock:
        log.warning("only %s available, %s requested", stock, wanted)
        return stock
    log.info("reserved %s items", wanted)
    return wanted

reserve(10, 3)
reserve(2, 5)

try:
    1 / 0
except ZeroDivisionError:
    log.exception("calculation failed")        # includes the traceback

| Level | Use it for |
|---|---|
| `DEBUG` | detail for developers |
| `INFO` | normal milestones |
| `WARNING` | something unexpected, but handled |
| `ERROR` | an operation failed |
| `CRITICAL` | the program may not continue |

Pass values as arguments (`log.info("x=%s", x)`) rather than pre-formatting; the string is built only if the message is emitted.

## 18.7 Tooling

In [ ]:
%%bash
python -m pip install ruff black mypy pytest

ruff check .          # fast linter: unused imports, likely bugs, style
black .               # auto-format to one consistent style
mypy src/             # static type checking using your hints
pytest                # run tests (finds test_*.py automatically)

A healthy project runs formatter, linter, type checker, and tests automatically — locally with a pre-commit hook, and on every push with continuous integration.

## 18.8 A pre-commit checklist

- [ ] Names describe intent
- [ ] Every public function has a docstring
- [ ] No unused imports or variables
- [ ] No mutable default arguments
- [ ] Errors are handled specifically, never silently swallowed
- [ ] Logic is covered by at least a few tests
- [ ] Secrets (passwords, tokens) are not in the code
- [ ] It works on a fresh virtual environment

## Common pitfalls

- Over-clever one-liners that save lines but cost readability.
- Comments that contradict the code after edits.
- Premature optimisation: measure with `time.perf_counter()` or `cProfile` before changing anything.
- Letting a function grow past a screen of code.

## Practice

1. Take any function from Modules 1–10 and rewrite it with a docstring, type hints, and clearer names.
2. Replace `print` statements in one of your scripts with `logging` at appropriate levels.
3. Install `ruff` and `black`, run them on a folder of your code, and fix everything they report.